# Antibody Developability Prediction from Sequence-Derived Features

Independent computational research project exploring prediction of antibody developability from sequence-derived and protein-language-model features.

## Workflow

1. Construct a developability label from experimentally measured assay attributes.
2. Remove non-feature, sequence, metadata and assay columns to avoid label leakage.
3. Perform L1 (LASSO) feature selection using the training set.
4. Evaluate selected features with L2 logistic regression.
5. Train a LightGBM classifier on the selected feature set.
6. Evaluate classification performance and inspect feature contributions using SHAP.

**Note:** This repository documents the computational workflow. The original experimental dataset is not included.


## Data

The original analysis used antibody-level data containing sequence-derived descriptors and experimentally measured developability attributes.

The working analysis contained approximately 246 antibodies. The feature set included sequence-derived descriptors and reduced protein-language-model representations, including ESM-derived and ProtBERT-derived principal components.

**The dataset is intentionally not included in this repository.** It must only be added locally if you have permission to use and redistribute it. See `data/README.md`.


## 1. L1 feature selection and L2 evaluation

L1-regularised logistic regression (LASSO) was used to select features on the training data. The non-zero features were then evaluated using an L2-regularised logistic regression model.

This preserves the feature-selection/evaluation workflow used in the working analysis.

In [ ]:
# l1_selector_then_eval_l2.py
import json
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss,
    roc_curve, auc, log_loss, mean_squared_error, r2_score, precision_recall_curve
)
import matplotlib.pyplot as plt

# ---------------- CONFIGURATION ----------------
CSV_PATH = "../data/descriptors_batch_all_merged.csv"
OUT_DIR = Path("../results/L1_L2_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
C_GRID_L1 = [0.001, 0.01, 0.1, 1.0, 10.0]   # for L1 selection
C_GRID_L2 = [0.01, 0.1, 1.0, 10.0]         # for L2 evaluation model
TOP_K = 20
MIN_NONNA_RATIO = 0.3
DEVELOPABLE_FRACTION_CUTOFF = 0.95

# ---------------- WEIGHTS and THRESHOLDS (provided) ----------------
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 1.0,
    "titer_productionbatch1_avg": 1.0,
    "purity_%lc+hc_avg": 1.0,
    "sec_%monomer_avg": 1.2,
    "tonset_nanodsf_avg": 1.0,
    "tm1_nanodsf_avg": 1.0,
    "tm2_nanodsf_avg": 1.0,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 1.2,
    "acsins_dLmax_ph7.4_avg": 1.2,
    "acsins_dLmax_ph6.0_avg": 1.2,
    "polyreactivity_prscore_cho_avg": 1.2,
    "polyreactivity_prscore_ova_avg": 1.2,
}

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 95.0, "medium": 90.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 55.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 60.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 70.0, "direction": "high"},
    "smac_rt_avg": {"high": 12.8, "medium": 15.0, "direction": "low"},
    "hic_rt_avg": {"high": 11.7, "medium": 14.0, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 20.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 20.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
}

# ---------------- Exact excludes (user insisted) ----------------
EXCLUDE_COLS = [
    "antibody_id", "antibody_name", "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025", "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean", "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean"
]
EXTRA_EXCLUDE = {"titer_productionbatch1_avg", "hac_rt_avg"}  # enforced
DERIVED_LABEL_COLS = {
    "developable_bin", "developability_fraction",
    "weighted_pass_sum", "available_weight_sum", "developability_index_unweighted",
    "developability_index", "developability_score"
}
EXCLUDE_PATTERNS = ["_pass", "_avg_pass", "_binary", "_flag", "pass_", "pass$", "_percent_pass"]

# ---------------- Load ----------------
df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH, "shape:", df.shape)

# ---------------- Build label ----------------
assay_keys = list(THRESHOLDS.keys())
weights_map = WEIGHTS.copy()
for k in assay_keys:
    if k not in weights_map:
        weights_map[k] = 1.0

weighted_scores = []
available_weights = []
found_any = False

for assay in assay_keys:
    if assay not in df.columns:
        print(f"WARNING: assay column {assay} not found; skipping it for label creation.")
        continue
    found_any = True
    thr = THRESHOLDS[assay]
    direction = thr["direction"]
    high = thr["high"]
    medium = thr["medium"]
    w = float(weights_map.get(assay, 1.0))
    col = df[assay]

    if direction == "high":
        score_vec = np.where(col >= high, 1.0, np.where(col >= medium, 0.5, 0.0))
    else:
        score_vec = np.where(col <= high, 1.0, np.where(col <= medium, 0.5, 0.0))

    score_series = pd.Series(score_vec, index=df.index).astype(float) * w
    avail_series = pd.Series(np.where(col.notna(), w, 0.0), index=df.index)

    weighted_scores.append(score_series)
    available_weights.append(avail_series)

if not found_any:
    raise RuntimeError("No assay columns found to construct developable_bin — aborting.")

weighted_pass_sum = pd.concat(weighted_scores, axis=1).sum(axis=1)
available_weight_sum = pd.concat(available_weights, axis=1).sum(axis=1)
with np.errstate(divide='ignore', invalid='ignore'):
    developability_fraction = np.where(available_weight_sum > 0, weighted_pass_sum / available_weight_sum, np.nan)

df["developability_fraction"] = developability_fraction
df["weighted_pass_sum"] = weighted_pass_sum
df["available_weight_sum"] = available_weight_sum
df["developable_bin"] = pd.Series(np.where(df["developability_fraction"] >= DEVELOPABLE_FRACTION_CUTOFF, 1, 0), index=df.index)
print("Derived developable_bin counts (including NaNs handled):\n", df["developable_bin"].value_counts(dropna=False))

# keep only rows where label exists
df = df[df["available_weight_sum"] > 0].copy()
n_rows = len(df)
print("Rows used:", n_rows)

# ---------------- Build candidate features (exclude assays etc) ----------------
exclude_set = set(THRESHOLDS.keys()) | set(EXCLUDE_COLS) | EXTRA_EXCLUDE | DERIVED_LABEL_COLS
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

def is_excluded_col(col):
    if col in exclude_set:
        return True
    low = col.lower()
    for pat in EXCLUDE_PATTERNS:
        if pat.replace("$","") in low:
            return True
    for t in THRESHOLDS.keys():
        if t in col and col != t:
            return True
    if "pass" in low:
        return True
    if col in DERIVED_LABEL_COLS or col in {"developable_bin", "developability_fraction"}:
        return True
    return False

candidate_features = []
for c in numeric_cols:
    if is_excluded_col(c):
        continue
    nonna_ratio = df[c].notna().sum() / float(n_rows)
    if nonna_ratio < MIN_NONNA_RATIO:
        continue
    candidate_features.append(c)

print("Candidate numeric features count:", len(candidate_features))
print("Example candidates (first 40):", candidate_features[:40])

if len(candidate_features) == 0:
    raise RuntimeError("No candidate features after exclusion. Adjust MIN_NONNA_RATIO or exclusions.")

# ---------------- Prepare X,y and split ----------------
X = df[candidate_features].copy()
y = df["developable_bin"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print("Train/test shapes:", X_train.shape, X_test.shape)

# ---------------- L1 selection on TRAIN only ----------------
preproc_sel = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
pipe_l1 = Pipeline([("preproc", preproc_sel), ("clf", LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))])
param_grid_l1 = {"clf__C": C_GRID_L1}
cv_sel = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
grid_l1 = GridSearchCV(pipe_l1, param_grid=param_grid_l1, cv=cv_sel, scoring="f1", n_jobs=-1, verbose=1)
grid_l1.fit(X_train, y_train)

print("L1 selection best params:", grid_l1.best_params_, "best CV F1:", grid_l1.best_score_)

best_l1 = grid_l1.best_estimator_
coefs = best_l1.named_steps["clf"].coef_.ravel()
feat_rank_df = pd.DataFrame({"feature": candidate_features, "coef": coefs, "abs_coef": np.abs(coefs)})
feat_rank_df = feat_rank_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
feat_rank_df.to_csv(OUT_DIR / "l1_feature_ranking_full.csv", index=False)
feat_rank_df.head(TOP_K).to_csv(OUT_DIR / f"l1_top_{TOP_K}.csv", index=False)
feat_rank_df[feat_rank_df["abs_coef"] > 1e-8].to_csv(OUT_DIR / "l1_nonzero_selected_features.csv", index=False)
print("Saved L1 ranking and selected lists.")

# ---------------- Train an evaluation model (L2 Logistic) on selected features (train) ----------------
selected_nonzero = feat_rank_df[feat_rank_df["abs_coef"] > 1e-8]["feature"].tolist()
# If none non-zero, fall back to top-K
if len(selected_nonzero) == 0:
    selected_nonzero = feat_rank_df.head(TOP_K)["feature"].tolist()
print("Selected features used for evaluation (count):", len(selected_nonzero))

X_train_sel = X_train[selected_nonzero].copy()
X_test_sel = X_test[selected_nonzero].copy()

# L2 grid on selected features
preproc_eval = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
pipe_l2 = Pipeline([("preproc", preproc_eval), ("clf", LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))])
param_grid_l2 = {"clf__C": C_GRID_L2}
cv_eval = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
grid_l2 = GridSearchCV(pipe_l2, param_grid=param_grid_l2, cv=cv_eval, scoring="f1", n_jobs=-1, verbose=1)
grid_l2.fit(X_train_sel, y_train)

best_l2 = grid_l2.best_estimator_
print("L2 eval best params:", grid_l2.best_params_, "best CV F1:", grid_l2.best_score_)

# ---------------- Predictions on test ----------------
y_prob = best_l2.predict_proba(X_test_sel)[:,1]
y_pred = best_l2.predict(X_test_sel)

# Basic metrics
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "pr_auc": float(average_precision_score(y_test, y_prob)),
    "brier_loss": float(brier_score_loss(y_test, y_prob)),
    "logloss": float(log_loss(y_test, y_prob)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}
with open(OUT_DIR / "evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved evaluation metrics:", metrics)

# ---------------- Extended metrics: R2 / RMSE / top20 ----------------
r2 = float(r2_score(y_test, y_prob))
mse = mean_squared_error(y_test, y_prob)
rmse = float(np.sqrt(mse))

def top_k_metrics(y_true, y_prob, k=20):
    n = len(y_prob)
    if n == 0:
        return {"r2_topk": None, "rmse_topk": None}
    idx = np.argsort(-y_prob)
    k2 = min(k, n)
    top_idx = idx[:k2]
    r2_top = float(r2_score(y_true.iloc[top_idx], y_prob[top_idx])) if k2>0 else None
    mse_top = mean_squared_error(y_true.iloc[top_idx], y_prob[top_idx])
    rmse_top = float(np.sqrt(mse_top)) if k2>0 else None
    return {"r2_topk": r2_top, "rmse_topk": rmse_top}

top20 = top_k_metrics(y_test.reset_index(drop=True), y_prob, k=20)

extended = {"r2": r2, "rmse": rmse, "top20": top20}
with open(OUT_DIR / "evaluation_extended_metrics.json", "w") as f:
    json.dump(extended, f, indent=2)

# ---------------- Plots ----------------
# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"L2 (AUC = {roc_auc:.3f})")
plt.plot([0,1],[0,1],'--', linewidth=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Eval model on selected features)")
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_curve_eval.png", dpi=150)
plt.close()

# PR curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
plt.figure(figsize=(7,6))
plt.plot(rec, prec, label=f"PR (AP = {metrics['pr_auc']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Eval model)")
plt.legend(loc="lower left")
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "pr_curve_eval.png", dpi=150)
plt.close()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="viridis")
for (i,j), val in np.ndenumerate(cm):
    ax.text(j, i, val, ha="center", va="center", color="w", fontsize=12)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Eval model)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_eval.png", dpi=150)
plt.close()

# Metric comparison bar
labels = ["Accuracy", "F1", "Precision", "Recall", "ROC AUC", "PR AUC", "Brier", "LogLoss"]
vals = [metrics["accuracy"], metrics["f1"], metrics["precision"], metrics["recall"], metrics["roc_auc"], metrics["pr_auc"], metrics["brier_loss"], metrics["logloss"]]
x = np.arange(len(labels))
width = 0.6
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x, vals, width)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_title("Evaluation Metrics (selected features)")
plt.tight_layout()
plt.savefig(OUT_DIR / "metrics_comparison_eval.png", dpi=150)
plt.close()

# Calibration scatter: prob vs true (jitter)
plt.figure(figsize=(6,4))
plt.scatter(y_prob, y_test + (np.random.randn(len(y_test)) * 0.02), alpha=0.7, s=20)
plt.xlabel("Predicted probability")
plt.ylabel("True label (jittered)")
plt.title("Predicted prob vs true label (jitter)")
plt.tight_layout()
plt.savefig(OUT_DIR / "prob_vs_true_scatter_eval.png", dpi=150)
plt.close()

# Learning curve (train sizes vs CV F1 for eval model)
train_sizes, train_scores, test_scores = learning_curve(
    grid_l2.best_estimator_, X_train_sel, y_train, cv=cv_eval, scoring="f1",
    train_sizes=np.linspace(0.2,1.0,5), n_jobs=-1, random_state=RANDOM_STATE
)
train_scores_mean = np.mean(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_scores_mean, "o-", label="Train F1")
plt.plot(train_sizes, test_scores_mean, "o-", label="CV F1")
plt.xlabel("Training examples")
plt.ylabel("F1 score")
plt.title("Learning Curve (Eval model)")
plt.legend()
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curve_eval.png", dpi=150)
plt.close()

# Save predictions with probs
preds_df = pd.DataFrame({
    "index": X_test_sel.index,
    "developable_true": y_test.values,
    "prob": y_prob,
    "pred": y_pred
})
preds_df.to_csv(OUT_DIR / "eval_test_predictions.csv", index=False)

# Save summary
summary = {
    "selected_features_count": len(selected_nonzero),
    "selected_features": selected_nonzero,
    "evaluation_metrics": metrics,
    "extended": extended,
    "l1_best_C": grid_l1.best_params_["clf__C"],
    "l2_best_C": grid_l2.best_params_["clf__C"]
}
with open(OUT_DIR / "l1_selection_eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("All diagnostics saved to:", OUT_DIR)


## 2. Final LightGBM model

The later/final LightGBM workflow in the supplied notebook:

- derives the binary developability label from weighted assay pass criteria;
- excludes assay values and derived label columns from the ML feature matrix;
- applies median imputation;
- uses stratified 5-fold cross-validation;
- performs LightGBM hyperparameter search using average precision;
- calibrates the selected model;
- evaluates ROC-AUC, PR-AUC, precision, recall, F1, MCC and Brier loss;
- generates SHAP feature-importance summaries and diagnostic plots.

The code below is retained from the later LightGBM version in the working notebook, with only the file paths made repository-relative.

In [ ]:
#!/usr/bin/env python3
"""
Fast developability training — Stage-1 only (LightGBM RandomizedSearch)
- Stage-2 REMOVED (no GridSearch / early stopping) to avoid Jupyter flooding issues.
- n_iter = 6 for now (short debugging mode). Can be increased later.
- Outputs: metrics, cv_results, predictions, plots, feature importances, threshold sweep.

MODIFICATION: Medium tier removed from pass criteria. An assay only "passes"
if it meets the 'high' threshold. (tier_for_value now returns only "High"/"Low"
and the _pass mapping treats only "High" as pass; NaNs are preserved.)
"""

import os
import json
import joblib
from pathlib import Path
from datetime import datetime, UTC
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# sklearn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, learning_curve
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef, brier_score_loss,
    precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
import shap
from sklearn.metrics import mean_squared_error, r2_score
# LightGBM
import lightgbm as lgb

# quiet warnings
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_style("whitegrid")

# ---------------- CONFIG ----------------
INPUT_CSV = "../data/descriptors_batch_all_merged_nonzero.csv"
BASE_OUTDIR = Path("../results/lightgbm")
RUN_PREFIX = "run"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_TOP_FEATURES = 25

 # weights / thresholds
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 0.8,
    "titer_productionbatch1_avg": 0.8,
    "purity_%lc+hc_avg": 1.2,
    "sec_%monomer_avg": 1.3,
    "tonset_nanodsf_avg": 1.5,
    "tm1_nanodsf_avg": 1.5,
    "tm2_nanodsf_avg": 1.5,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 0.0,
    "acsins_dLmax_ph7.4_avg": 0.8,
    "acsins_dLmax_ph6.0_avg": 0.8,
    "polyreactivity_prscore_cho_avg": 1.0,
    "polyreactivity_prscore_ova_avg": 1.0,
}

DEVELOPABILITY_THRESHOLD = 0.82

EXCLUDE_COLS = [
    "antibody_id", "antibody_name",
    "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025",
    "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean",
    "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean",
    # assay columns intentionally not listed here (they're handled separately)
]

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 150.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 150.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 98.0, "medium": 96.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 60.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 75.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 80.0, "direction": "high"},
    "smac_rt_avg": {"high": 2.0, "medium": 3.0, "direction": "low"},
    "hic_rt_avg": {"high": 2.0, "medium": 3.5, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 13.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 13.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.29, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.29, "direction": "low"},
}

ASSAYS = list(THRESHOLDS.keys())
SOFT_ASSAYS = ["normalized_titer_productionbatch1_avg", "titer_productionbatch1_avg"]

# ---------------- helpers ----------------
def next_run_folder(base_dir: Path, prefix="run"):
    """Ensure run001, run002, run003… correct numeric order."""
    base_dir.mkdir(parents=True, exist_ok=True)
    nums = []
    for p in base_dir.iterdir():
        if p.is_dir() and p.name.startswith(prefix):
            suffix = p.name.replace(prefix, "")
            if suffix.isdigit():
                nums.append(int(suffix))
    new_num = max(nums) + 1 if nums else 1
    return base_dir / f"{prefix}{new_num}"

def safe_savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

# ---------------- MODIFIED: tier logic (NO MEDIUM) ----------------
def tier_for_value(val, thr):
    """
    Return only "High" or "Low" (or np.nan for missing).
    An assay is 'High' only if it meets the 'high' threshold.
    'medium' entries in THRESHOLDS are ignored for pass/tier.
    """
    if pd.isna(val):
        return np.nan

    if thr["direction"] == "high":
        return "High" if val >= thr["high"] else "Low"
    else:
        return "High" if val <= thr["high"] else "Low"

# ---------------- LOAD DATA ----------------
if not Path(INPUT_CSV).exists():
    raise FileNotFoundError(f"Missing: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

# ---------------- TIERS & PASSES ----------------
for assay, thr in THRESHOLDS.items():
    if assay not in df.columns:
        df[assay] = np.nan
    # compute tier (now only High/Low/NaN)
    df[f"{assay}_tier"] = df[assay].apply(lambda v: tier_for_value(v, thr))
    # compute pass: only High => 1, Low => 0, preserve NaN where tier is NaN
    df[f"{assay}_pass"] = df[f"{assay}_tier"].map({"High": 1, "Low": 0})
    df.loc[df[f"{assay}_tier"].isna(), f"{assay}_pass"] = np.nan

# ---------------- WEIGHTED DEVELOPABILITY ----------------
pass_cols = [f"{a}_pass" for a in ASSAYS]

# --- SAFE WEIGHT VECTOR: default to 1.0 if key missing (won't override provided weights) ---
weight_vec = np.array([WEIGHTS.get(a, 1.0) for a in ASSAYS], float)

passes_matrix = df[pass_cols].to_numpy(float)
available_mask = ~np.isnan(passes_matrix)

weighted_pass = np.nansum(np.where(available_mask, passes_matrix * weight_vec, 0.0), axis=1)
available_weight = np.sum(np.where(available_mask, weight_vec, 0.0), axis=1)
weighted_fraction = np.where(available_weight > 0, weighted_pass / available_weight, np.nan)

df["weighted_pass_sum"] = weighted_pass
df["available_weight_sum"] = available_weight
df["developability_fraction"] = weighted_fraction
df["developability_index_unweighted"] = df[pass_cols].sum(axis=1, skipna=True)
df["developable_bin"] = (df["developability_fraction"] >= DEVELOPABILITY_THRESHOLD).astype(int)
df.loc[df["developability_fraction"].isna(), "developable_bin"] = np.nan

# ---------------- FEATURES FOR ML ----------------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
tier_cols = [c for c in df.columns if c.endswith("_tier")]
pass_cols_actual = [c for c in df.columns if c.endswith("_pass")]

cols_to_remove = (
    set(tier_cols) |
    set(pass_cols_actual) |
    set(EXCLUDE_COLS) |
    set(ASSAYS) |
    {"developable_bin", "developability_fraction", "developability_index_unweighted",
     "weighted_pass_sum", "available_weight_sum"}
)

feature_cols = [c for c in numeric_cols if c not in cols_to_remove]
feat_missing_frac = df[feature_cols].isna().mean() if feature_cols else pd.Series(dtype=float)
feature_cols = [c for c in feature_cols if feat_missing_frac[c] <= 0.5]

print(f"LEAK-FREE FEATURE COUNT: {len(feature_cols)}")

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(df[feature_cols].values)

y_series = df["developable_bin"]
has_label_mask = ~y_series.isna()
y_all = y_series[has_label_mask].astype(int).to_numpy()
X_all_labeled = X_all[has_label_mask.values, :]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all_labeled, y_all, df.index.values[has_label_mask.values],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

# ---------------- RUN DIR ----------------
run_dir = next_run_folder(BASE_OUTDIR, RUN_PREFIX)
run_dir.mkdir(parents=True, exist_ok=True)

config = {
    "input_file": INPUT_CSV,
    "date": "Omitted by design",
    "thresholds": THRESHOLDS,
    "weights": WEIGHTS,
    "developability_threshold_value": DEVELOPABILITY_THRESHOLD,
    "n_features": len(feature_cols),
    "feature_columns": feature_cols
}

with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(run_dir / "run_config.txt", "w") as f:
    f.write("=== STAGE-1 ONLY: RANDOMIZED SEARCH (LightGBM) ===\n")
    f.write(f"Run Directory: {run_dir}\nDate (UTC): {datetime.now(UTC).isoformat()}\n")
    f.write(f"Features used: {len(feature_cols)}\n")

# ---------------- RANDOMIZED SEARCH ----------------
print("Starting Stage-1 RandomizedSearchCV (LightGBM)")

param_distributions_lgb = {
    "n_estimators": [200],
    "learning_rate": [0.1],
    "num_leaves": [16],
    "max_depth": [4],
    "min_child_samples": [10],
    "subsample": [0.6],
    "colsample_bytree": [0.6],
    "reg_alpha": [0.0],
    "reg_lambda": [0.1],
    "class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

try:
    lgb_est = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        device= "gpu",
        class_weight="balanced"
    )
    print("Using GPU for LightGBM")
except:
    lgb_est = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        device= "cpu",
        class_weight="balanced"
    )
    print("GPU unavailable → using CPU")

random_search = RandomizedSearchCV(
    estimator=lgb_est,
    param_distributions=param_distributions_lgb,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    refit=True,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

# calibrated model
base_model = random_search.best_estimator_
clf = CalibratedClassifierCV(base_model, cv=5, method="sigmoid")
clf.fit(X_train, y_train)

best_params = random_search.best_params_

# update config
config["algorithm"] = "LightGBM"
config["best_hyperparameters_lgb"] = best_params
config["best_cv_score_stage1_average_precision"] = float(random_search.best_score_)
with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

pd.DataFrame(random_search.cv_results_).to_csv(run_dir / "cv_results_stage1.csv", index=False)

joblib.dump({"model": clf, "features": feature_cols, "imputer": imputer},
            str(run_dir / "lgb_stage1_model.joblib"))

print("Stage-1 complete. Best params:", best_params)
print("Best CV (average_precision):", random_search.best_score_)

# ---------------- METRICS & PREDICTIONS ----------------
y_proba = clf.predict_proba(X_test)[:, 1]

try:
    shap_sample = X_test if X_test.shape[0] <= 500 else X_test[np.random.RandomState(RANDOM_STATE).choice(X_test.shape[0], 500, replace=False)]
    explainer = shap.TreeExplainer(base_model)
    shap_values = explainer.shap_values(shap_sample)

    if isinstance(shap_values, list) and len(shap_values) > 1:
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    plt.figure(figsize=(8,6))
    shap.summary_plot(shap_vals, shap_sample, feature_names=feature_cols, show=False)
    safe_savefig(plt.gcf(), run_dir / "shap_summary.png")

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs_shap})
    shap_df = shap_df.sort_values("mean_abs_shap", ascending=False)
    shap_df.to_csv(run_dir / "shap_mean_abs_values.csv", index=False)

except Exception as e_shap:
    print("SHAP computation failed:", str(e_shap))

# 2) RMSE and R2 on probabilities
try:
    rmse_prob = float(np.sqrt(mean_squared_error(y_test, y_proba)))
    r2_prob = float(r2_score(y_test, y_proba))
except:
    rmse_prob, r2_prob = float("nan"), float("nan")

# will merge into main metrics object below
extra_reg_metrics = {
    "rmse_prob_vs_label": rmse_prob,
    "r2_prob_vs_label": r2_prob
}
# PR curve sweep BEFORE choosing cutoff
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_test, y_proba)
f1_arr = [0.0 if (p + r) == 0 else 2 * p * r / (p + r)
          for p, r in zip(precision_arr[:-1], recall_arr[:-1])]

pr_sweep_df = pd.DataFrame({
    "threshold": pr_thresholds,
    "precision": precision_arr[:-1],
    "recall": recall_arr[:-1],
    "f1": f1_arr
})
pr_sweep_df.to_csv(run_dir / "threshold_sweep_prcurve.csv", index=False)

# choose operational cutoffs
MIN_PREC = 0.60
candidates = pr_sweep_df[pr_sweep_df["precision"] >= MIN_PREC]

if not candidates.empty:
    cutoff_recall = float(candidates.sort_values("recall", ascending=False).iloc[0]["threshold"])
else:
    cutoff_recall = float(pr_sweep_df.sort_values("recall", ascending=False).iloc[0]["threshold"])

cutoff_f1 = float(pr_sweep_df.sort_values("f1", ascending=False).iloc[0]["threshold"])

print("\nTop 5 thresholds (by recall):")
print(pr_sweep_df.sort_values("recall", ascending=False).head())
print("\nTop 5 thresholds (by F1):")
print(pr_sweep_df.sort_values("f1", ascending=False).head())

print(f"\nSelected cutoff_recall: {cutoff_recall:.4f}")
print(f"Selected cutoff_f1: {cutoff_f1:.4f}")

# apply chosen cutoff for main metrics
# = (y_proba >= cutoff_recall).astype(int)
y_pred = (y_proba >= cutoff_f1).astype(int)

# save metrics
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "pr_auc": float(average_precision_score(y_test, y_proba)),
    "mcc": float(matthews_corrcoef(y_test, y_pred)),
    "brier_loss": float(brier_score_loss(y_test, y_proba))
}

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
# include extra reg metrics
metrics.update(extra_reg_metrics)

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# predictions file
# create preds_df for test rows (fixing typo and ordering)
preds_df = df.loc[idx_test, ["antibody_id", "antibody_name"] + ASSAYS].copy().reset_index(drop=True)

# attach true/pred/prob for the test set
preds_df["developable_true"] = y_test
preds_df["developable_pred"] = y_pred
preds_df["developable_prob"] = y_proba

# attach developability summary columns for test rows
preds_df["developability_fraction"] = df.loc[idx_test, "developability_fraction"].values
preds_df["available_weight_sum"] = df.loc[idx_test, "available_weight_sum"].values
preds_df["weighted_pass_sum"] = df.loc[idx_test, "weighted_pass_sum"].values

# save the full processed dataframe (all 246 rows) once
df.to_csv("../data/descriptors_batch_all_merged.csv", index=False)

# save the test-set predictions (as before)
preds_df.to_csv(run_dir / "predictions_test.csv", index=False)
print("Saved test-set predictions:", run_dir / "predictions_test.csv")


# ---------------- PLOTS ----------------

# ROC
if len(np.unique(y_test)) == 2:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "roc_curve.png")

# PR
if len(np.unique(y_test)) == 2:
    ap = average_precision_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(recall_arr, precision_arr, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "pr_curve.png")

# Confusion Matrix
plt.figure(figsize=(4,3))
sns.heatmap(np.array([[tn, fp],[fn, tp]]), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
safe_savefig(plt.gcf(), run_dir / "confusion_matrix.png")

# Calibration
if len(np.unique(y_test)) == 2:
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
    plt.figure(figsize=(6,5))
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("Mean predicted probability"); plt.ylabel("Fraction of positives")
    plt.title("Calibration plot"); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "calibration.png")

# --- ADDED: ACCURACY vs PROBABILITY CUTOFF SWEEP ---
try:
    acc_thresholds = np.linspace(0.0, 1.0, 101)
    acc_vals = []
    prec_vals = []
    rec_vals = []
    f1_vals = []
    for thr in acc_thresholds:
        y_thr = (y_proba >= thr).astype(int)
        acc_vals.append(accuracy_score(y_test, y_thr))
        prec_vals.append(precision_score(y_test, y_thr, zero_division=0))
        rec_vals.append(recall_score(y_test, y_thr, zero_division=0))
        f1_vals.append(f1_score(y_test, y_thr, zero_division=0))
    acc_df = pd.DataFrame({
        "threshold": acc_thresholds,
        "accuracy": acc_vals,
        "precision": prec_vals,
        "recall": rec_vals,
        "f1": f1_vals
    })
    acc_df.to_csv(run_dir / "accuracy_vs_cutoff.csv", index=False)

    plt.figure(figsize=(8,5))
    plt.plot(acc_thresholds, acc_vals, label="accuracy")
    plt.plot(acc_thresholds, prec_vals, label="precision")
    plt.plot(acc_thresholds, rec_vals, label="recall")
    plt.plot(acc_thresholds, f1_vals, label="f1")
    plt.xlabel("Probability cutoff"); plt.ylabel("Metric"); plt.title("Accuracy/Precision/Recall/F1 vs cutoff")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "accuracy_precision_recall_f1_vs_cutoff.png")
except Exception as e_acc:
    print("Accuracy vs cutoff sweep failed:", str(e_acc))

# --- ADDED: LEARNING CURVE (train/validation) ---
try:
    train_sizes = np.linspace(0.1, 1.0, 5)
    # use base_model (not calibrated) to get learning behaviour
    train_sizes_abs, train_scores, val_scores = learning_curve(
        base_model, X_all_labeled, y_all, cv=cv, train_sizes=train_sizes,
        scoring="accuracy", n_jobs=-1, random_state=RANDOM_STATE
    )
    train_scores_mean = train_scores.mean(axis=1)
    train_scores_std = train_scores.std(axis=1)
    val_scores_mean = val_scores.mean(axis=1)
    val_scores_std = val_scores.std(axis=1)

    lc_df = pd.DataFrame({
        "train_size": train_sizes_abs,
        "train_score_mean": train_scores_mean,
        "train_score_std": train_scores_std,
        "val_score_mean": val_scores_mean,
        "val_score_std": val_scores_std
    })
    lc_df.to_csv(run_dir / "learning_curve_accuracy.csv", index=False)

    plt.figure(figsize=(7,5))
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1)
    plt.fill_between(train_sizes_abs, val_scores_mean - val_scores_std, val_scores_mean + val_scores_std, alpha=0.1)
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', label='Train score')
    plt.plot(train_sizes_abs, val_scores_mean, 'o-', label='Validation score')
    plt.xlabel("Training examples"); plt.ylabel("Accuracy"); plt.title("Learning curve (accuracy)")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "learning_curve_accuracy.png")
except Exception as e_lc:
    print("Learning curve computation failed:", str(e_lc))

# --- ADDED: CLASS DISTRIBUTION PLOT ---
try:
    vc = df["developable_bin"].value_counts(dropna=False).sort_index()
    plt.figure(figsize=(5,4))
    sns.barplot(x=vc.index.astype(str), y=vc.values)
    plt.xlabel("developable_bin"); plt.ylabel("count"); plt.title("Class distribution (full df)")
    safe_savefig(plt.gcf(), run_dir / "class_distribution_full.png")
    vc.loc[~df.loc[has_label_mask, 'developable_bin'].isna()].to_frame().to_csv(run_dir / "class_distribution_counts.csv")
except Exception as e_cd:
    print("Class distribution plot failed:", str(e_cd))

# --- ADDED: FEATURE CORRELATION HEATMAP ---
try:
    if len(feature_cols) >= 2:
        corr = df[feature_cols].corr(method="pearson")
        plt.figure(figsize=(max(6, 0.25*len(feature_cols)), max(6, 0.25*len(feature_cols))))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=False, cmap="vlag", center=0)
        plt.title("Feature correlation (pearson)")
        safe_savefig(plt.gcf(), run_dir / "feature_correlation_heatmap.png")
        corr.to_csv(run_dir / "feature_correlation_matrix.csv")
except Exception as e_corr:
    print("Feature correlation heatmap failed:", str(e_corr))
# --- END ADDED BLOCKS ---

# Feature Importances (horizontal barplot for readability)
fi = base_model.feature_importances_
fi_idx = np.argsort(fi)[::-1][:N_TOP_FEATURES]
top_features = [(feature_cols[i], float(fi[i])) for i in fi_idx]
fi_df = pd.DataFrame(top_features, columns=["feature", "importance"])
fi_df.to_csv(run_dir / "feature_importances_top.csv", index=False)

plt.figure(figsize=(8, max(4, 0.25*len(fi_idx))))
sns.barplot(y=[n for n,_ in top_features], x=[f for _,f in top_features], orient='h')
plt.xlabel("Importance"); plt.ylabel("Feature")
plt.title("Top feature importances (LGBM)")
safe_savefig(plt.gcf(), run_dir / "feature_importances_top.png")

# Diagnostics
diag = {
    "n_total_rows": int(df.shape[0]),
    "n_rows_with_label": int(has_label_mask.sum()),
    "n_features_used": len(feature_cols),
    "developability_threshold_used": DEVELOPABILITY_THRESHOLD,
}
with open(run_dir / "diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"\nRUN COMPLETE → {run_dir}")
print("Stage-1 only run finished (Stage-2 omitted).")


## Interpretation

The model should be treated as an exploratory developability-prediction analysis rather than a validated production model. The dataset is relatively small, and performance estimates depend on label construction, feature selection and the train/test split.

The notebook documents the computational workflow used during the project. Numerical results should only be reported from saved result files that can be traced to the corresponding analysis run.
